# Recipe Archaeologist - Prototype Notebook
This notebook contains the complete logic for analyzing food images, inferring ingredients, and generating recipes.

In [ ]:
import cv2
import numpy as np
import json
from google import genai
from google.genai import types

## 1. Image Analysis
Extracts visual features from an image to inform ingredient deduction.

In [ ]:
class ImageAnalyzer:
    def __init__(self, image_path):
        self.image_path = image_path
        self.image = None
        self.image_rgb = None
        self.gray = None
        self.hsv = None

    # Load Image
    def load_image(self):
        self.image = cv2.imread(self.image_path)

        if self.image is None:
            raise ValueError("Image not found. Check file path.")

        self.image_rgb = cv2.cvtColor(self.image, cv2.COLOR_BGR2RGB)
        self.gray = cv2.cvtColor(self.image, cv2.COLOR_BGR2GRAY)
        self.hsv = cv2.cvtColor(self.image, cv2.COLOR_BGR2HSV)

    # Mean Color (RGB)
    def extract_mean_color(self):
        return np.mean(self.image_rgb, axis=(0, 1))

    # Color Variance
    def extract_color_variance(self):
        return np.var(self.image_rgb, axis=(0, 1))

    # Dominant Hue (HSV Space)
    def extract_dominant_hue(self):
        hue_channel = self.hsv[:, :, 0]
        return int(np.mean(hue_channel))

    # Oil Detection (Improved Logic)
    def detect_oil_presence(self):
        saturation = self.hsv[:, :, 1]
        value = self.hsv[:, :, 2]

        mean_saturation = np.mean(saturation)
        std_brightness = np.std(value)

        # Heuristic logic:
        oil_score = 0

        if mean_saturation < 80:
            oil_score += 1

        if std_brightness > 35:
            oil_score += 1

        oil_present = oil_score >= 1

        return oil_present, float(mean_saturation), float(std_brightness)

    # Edge Density (Texture Indicator)
    def analyze_texture(self):
        edges = cv2.Canny(self.gray, 100, 200)
        edge_density = np.sum(edges > 0) / edges.size

        if edge_density < 0.02:
            texture = "smooth"
        elif edge_density < 0.05:
            texture = "semi-grainy"
        else:
            texture = "grainy"

        return texture, float(edge_density)

    # Stain Area Ratio
    def calculate_stain_area_ratio(self):
        _, thresh = cv2.threshold(self.gray, 200, 255, cv2.THRESH_BINARY_INV)
        stain_pixels = np.sum(thresh > 0)
        total_pixels = thresh.size

        area_ratio = stain_pixels / total_pixels
        return float(area_ratio)

    # Spread Metric (Contour Analysis)
    def calculate_spread_metric(self):
        _, thresh = cv2.threshold(self.gray, 200, 255, cv2.THRESH_BINARY_INV)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            return 0.0

        largest_contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_contour)
        perimeter = cv2.arcLength(largest_contour, True)

        if perimeter == 0:
            return 0.0

        spread_metric = area / perimeter
        return float(spread_metric)

    # Full Analysis
    def analyze(self):
        self.load_image()

        mean_color = self.extract_mean_color()
        color_variance = self.extract_color_variance()
        dominant_hue = self.extract_dominant_hue()

        oil_presence, mean_saturation, std_brightness = self.detect_oil_presence()
        texture, edge_density = self.analyze_texture()

        stain_area_ratio = self.calculate_stain_area_ratio()
        spread_metric = self.calculate_spread_metric()

        features = {
            "mean_rgb": mean_color.tolist(),
            "color_variance": color_variance.tolist(),
            "dominant_hue": dominant_hue,
            "oil_presence": oil_presence,
            "mean_saturation": mean_saturation,
            "brightness_std": std_brightness,
            "edge_density": edge_density,
            "texture": texture,
            "stain_area_ratio": stain_area_ratio,
            "spread_metric": spread_metric
        }

        return features


## 2. Molecular Inference
Uses visual features to infer probable ingredients with rule-based scoring.

In [ ]:
class MolecularInference:
    def __init__(self, features):
        self.features = features
        self.scores = {}

    # Utility function
    def add_score(self, ingredient, value):
        if ingredient not in self.scores:
            self.scores[ingredient] = 0
        self.scores[ingredient] += value

    # Rule-Based Scoring
    def apply_color_rules(self):
        hue = self.features.get("dominant_hue", 0)
        mean_rgb = self.features.get("mean_rgb", [0, 0, 0])
        
        if hue < 15 or hue > 160:
            color = "dark-brown" if sum(mean_rgb) < 250 else "red-dominant"
        elif 15 <= hue <= 60:
            color = "dark-brown" if sum(mean_rgb) < 250 else "yellow-orange"
        else:
            color = "dark-brown" if sum(mean_rgb) < 250 else "yellow-orange"

        if color == "yellow-orange":
            self.add_score("turmeric", 0.6)
            self.add_score("red_chili_powder", 0.4)

        elif color == "red-dominant":
            self.add_score("red_chili_powder", 0.7)
            self.add_score("tomato_puree", 0.5)

        elif color == "dark-brown":
            self.add_score("caramelized_onion", 0.6)
            self.add_score("soy_sauce", 0.4)

    def apply_oil_rules(self):
        if self.features["oil_presence"]:
            self.add_score("refined_oil", 0.6)
            self.add_score("ghee", 0.5)

    def apply_texture_rules(self):
        texture = self.features["texture"]

        if texture == "grainy":
            self.add_score("ground_spices", 0.6)
            self.add_score("lentil_base", 0.5)

        elif texture == "semi-grainy":
            self.add_score("ground_spices", 0.4)

        elif texture == "smooth":
            self.add_score("cream", 0.5)
            self.add_score("yogurt", 0.5)

    # Normalize Scores (0–1)
    def normalize_scores(self):
        if not self.scores:
            return self.scores

        max_score = max(self.scores.values())

        for ingredient in self.scores:
            self.scores[ingredient] = round(
                self.scores[ingredient] / max_score, 2
            )

        return self.scores

    # Full Inference Pipeline
    def infer(self):
        self.apply_color_rules()
        self.apply_oil_rules()
        self.apply_texture_rules()
        return self.normalize_scores()


## 3. Recipe Generator
Uses LLM to reconstruct the dish based on features and inferred ingredients.

In [ ]:
class RecipeGenerator:
    def __init__(self, api_key):
        # The new SDK uses a centralized Client
        self.client = genai.Client(api_key=api_key)
        # Using Gemini 2.0 Flash (latest recommended for speed/JSON)
        self.model_id = "gemini-2.5-flash-lite"

    def build_prompt(self, features, ingredient_scores):
        # Prompt logic remains the same
        return f"""
You are a scientific culinary reconstruction AI.
You are reconstructing dishes based on chemical and visual residue evidence.

--- VISUAL FEATURES ---
Mean RGB: {features.get('mean_rgb')}
Color Variance: {features.get('color_variance')}
Dominant Hue (HSV scale): {features.get('dominant_hue')}
Mean Saturation: {features.get('mean_saturation')}
Oil Presence: {features.get('oil_presence')}
Brightness Std: {features.get('brightness_std')}
Edge Density: {features.get('edge_density')}
Texture Classification: {features.get('texture')}
Stain Area Ratio: {features.get('stain_area_ratio')}
Spread Metric: {features.get('spread_metric')}

--- INFERRED INGREDIENT SCORES ---
{json.dumps(ingredient_scores, indent=2)}

--- TASKS ---
1. Interpret what the hue, saturation, and brightness imply chemically.
2. Interpret what oil presence and spread imply about fat content.
3. Interpret what edge density and texture imply about solid particles.
4. Propose the TOP 3 most plausible dish categories.
5. Generate a realistic modern recipe for each dish.
6. Explain the molecular reasoning clearly.
7. Assign a confidence score (0-100) for each hypothesis.

Return ONLY valid JSON using this structure:

{{
  "hypotheses": [
    {{
      "dish_name": "string",
      "dish_category": "string",
      "ingredients": [
        {{"name": "string", "quantity": "string"}}
      ],
      "cooking_steps": ["string"],
      "molecular_reasoning": "string",
      "confidence_score": 0
    }}
  ]
}}
"""

    def generate_recipe(self, features, ingredient_scores):
        prompt = self.build_prompt(features, ingredient_scores)

        try:
            response = self.client.models.generate_content(
                model=self.model_id,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.6,
                    system_instruction="You are a molecular gastronomy reconstruction model."
                )
            )
            output_text = response.text
        except Exception as e:
            print(f"\n[Error] Gemini API Call Failed: {e}")
            output_text = json.dumps({
                "error": "API Error", 
                "dish_name": "Unknown", 
                "molecular_reasoning": str(e)
            })

        try:
            return json.loads(output_text)
        except Exception:
            return {"error": "Invalid JSON", "raw_output": output_text}


## 4. Run Analysis
Put it all together and run on an image.

In [ ]:
API_KEY = "YOUR_API_KEY_HERE"
image_path = "../backend/images/image1.png"

# Step 1: Image analysis
analyzer = ImageAnalyzer(image_path)
features = analyzer.analyze()

# Step 2: Molecular inference
inference = MolecularInference(features)
ingredient_scores = inference.infer()

# Step 3: LLM reconstruction
generator = RecipeGenerator(API_KEY)
recipe = generator.generate_recipe(features, ingredient_scores)

print("\nFINAL RECONSTRUCTED RECIPE -------")
print(json.dumps(recipe, indent=2))
